# Graphics-LPIPS QualCompare Revalidation

This notebook guides you through reproducing the QualCompare revalidation results using the pre-trained `TMQ_NR_8VP_yf03_kfolds` checkpoint.

## What you'll do:
1. Validate your rendered image directory structure
2. Run the Graphics-LPIPS metric evaluation
3. Compute correlation statistics and generate plots

## Prerequisites:
- Rendered image data prepared with QualCompare (see [README.md](README.md#expected-image-directory-hierarchy))
- Python dependencies installed: `pip install -r requirements.txt`
- PyTorch with GPU support (optional but recommended)

## Step 0: Configuration

Edit the paths below to match your local setup.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# === CONFIGURATION ===
# Set these to match your local setup

# Root directory where QualCompare outputs are stored
# Expected structure: SRC_ROOT/Source/<N>VP/... and SRC_ROOT/Distorted/<N>VP/...
# Use forward slashes (/) or Path for cross-platform compatibility
# Examples:
#   Linux/macOS:   "./rendered_data/TMQ/New_Render/Y_fixed_0.3"
#   Windows:       "./rendered_data/TMQ/New_Render/Y_fixed_0.3" (or r"C:\path\to\data")
#   Absolute path: str(Path.home() / "data/TMQ/New_Render/Y_fixed_0.3")
SRC_ROOT = "./rendered_data/TMQ/New_Render/Y_fixed_0.3"  # TODO: Change this path

# Dataset name (used for output organization)
DATABASE = "TMQ"

# Model checkpoint name (pre-trained)
MODEL_NAME = "TMQ_NR_8VP_yf03_kfolds"

# Number of views
VIEWS = 8

# Rendering parameters (must match your QualCompare rendering setup)
VIEW_METHOD = "Y_fixed_0.3"  # Camera height offset
RENDER_METHOD = "New_Render"

# CSV files with MOS scores and test lists
MOS_CSV = "./dataset/TMQ/TMQ_MOS.csv"
TESTLIST_CSV = "./dataset/TMQ/folds/TexturedDB_20_TestList_withnbPatchesPerVP_threth0.6.csv"

# Use folds when evaluating k-fold checkpoints (recommended for TMQ_NR_8VP_yf03_kfolds)
USE_FOLDS = True

# Output directory for results
OUT_ROOT = "./out"

# Use GPU if available
USE_GPU = True

# Repo root (where Graphics-LPIPS-QualCompare code is)
REPO_ROOT = Path.cwd()

print(f"Repository root: {REPO_ROOT}")
print(f"Render root: {SRC_ROOT}")
print(f"Output root: {OUT_ROOT}")
print(f"Model: {MODEL_NAME}")
print(f"Use folds: {USE_FOLDS}")

## Step 1: Validate Directory Structure

Check that your rendered data is organized correctly.

In [ ]:
def validate_render_structure(src_root, views):
    """
    Validate that the render directory has the expected structure.
    
    Expected:
    <src_root>/
      Source/<N>VP/obj1/views/*.png
      Source/<N>VP/obj1/patchs/*.csv
      Distorted/<N>VP>/obj_distorted/views/*.png
    """
    src_root = Path(src_root)
    issues = []
    
    if not src_root.exists():
        return False, [f"SRC_ROOT does not exist: {src_root}"]
    
    source_dir = src_root / "Source" / f"{views}VP"
    distorted_dir = src_root / "Distorted" / f"{views}VP"
    
    if not source_dir.exists():
        issues.append(f"Missing Source/{views}VP in {src_root}")
    else:
        ref_objs = [d.name for d in source_dir.iterdir() if d.is_dir()]
        if not ref_objs:
            issues.append(f"No objects found in Source/{views}VP")
        else:
            print(f"Found {len(ref_objs)} reference objects in Source/{views}VP")
            
            # Check structure of first object
            first_obj = source_dir / ref_objs[0]
            views_dir = first_obj / "views"
            patchs_dir = first_obj / "patchs"
            
            if not views_dir.exists():
                issues.append(f"Missing views/ in {first_obj.name}")
            else:
                png_count = len(list(views_dir.glob("*.png")))
                print(f"  └─ {ref_objs[0]}: {png_count} PNG files in views/")
            
            if not patchs_dir.exists():
                issues.append(f"Missing patchs/ in {first_obj.name}")
            else:
                csv_count = len(list(patchs_dir.glob("*.csv")))
                print(f"     └─ {csv_count} CSV files in patchs/")
    
    if not distorted_dir.exists():
        issues.append(f"Missing Distorted/{views}VP in {src_root}")
    else:
        dist_objs = [d.name for d in distorted_dir.iterdir() if d.is_dir()]
        if not dist_objs:
            issues.append(f"No objects found in Distorted/{views}VP")
        else:
            print(f"Found {len(dist_objs)} distorted objects in Distorted/{views}VP")
    
    return len(issues) == 0, issues

# Run validation
is_valid, issues = validate_render_structure(SRC_ROOT, VIEWS)

if is_valid:
    print("\n Render structure is valid!")
else:
    print("\n Issues found:")
    for issue in issues:
        print(f"  - {issue}")

## Step 2: Verify Checkpoint Availability

In [ ]:
# Check that the checkpoint is available
checkpoint_dir = REPO_ROOT / "checkpoints" / MODEL_NAME

if checkpoint_dir.exists():
    print(f" Checkpoint directory exists: {checkpoint_dir}")
    
    # List available folds
    folds = sorted([d.name for d in checkpoint_dir.iterdir() if d.is_dir() and d.name.startswith("fold_")])
    print(f"  Found {len(folds)} fold(s): {', '.join(folds)}")
    
    # Check for weights in each fold
    for fold in folds:
        weights = checkpoint_dir / fold / "latest_net_.pth"
        if weights.exists():
            size_mb = weights.stat().st_size / (1024*1024)
            print(f"    └─ {fold}: latest_net_.pth ({size_mb:.1f} MB)")
        else:
            print(f"       {fold}: missing latest_net_.pth")
else:
    print(f" Checkpoint not found: {checkpoint_dir}")

## Step 3: Run Evaluation

> Works for both single-checkpoint models and k-fold checkpoints (`USE_FOLDS=True`).

In [ ]:
# Create output directory
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

# Preflight checks for required files
required_paths = [
    ("MOS CSV", REPO_ROOT / MOS_CSV),
]
missing = [f"{label}: {path}" for label, path in required_paths if not path.exists()]

testlist_base = REPO_ROOT / TESTLIST_CSV
if USE_FOLDS:
    testlist_k0 = Path(str(testlist_base).replace(".csv", "_k0.csv"))
    if not testlist_k0.exists():
        missing.append(f"Fold test list CSV (k0): {testlist_k0}")
else:
    if not testlist_base.exists():
        missing.append(f"Test list CSV: {testlist_base}")

if USE_FOLDS:
    ckpt_k0 = REPO_ROOT / "checkpoints" / MODEL_NAME / "fold_k0" / "latest_net_.pth"
    if not ckpt_k0.exists():
        missing.append(f"Checkpoint fold k0: {ckpt_k0}")
else:
    ckpt_single = REPO_ROOT / "checkpoints" / MODEL_NAME / "latest_net_.pth"
    if not ckpt_single.exists():
        missing.append(f"Checkpoint: {ckpt_single}")

if missing:
    print("Missing required input files:")
    for item in missing:
        print(f"  - {item}")
    print("\nTip: for TMQ_NR_8VP_yf03_kfolds, set USE_FOLDS = True and TESTLIST_CSV without _k0 suffix.")
else:
    # Build evaluation command
    cmd = [
        sys.executable, "Light_GraphicsLPIPS_csv.py",
        "-m", MODEL_NAME,
        "-v", str(VIEWS),
        "-vm", VIEW_METHOD,
        "-rm", RENDER_METHOD,
        "-db", DATABASE,
        "-mos", MOS_CSV,
        "-testlist", TESTLIST_CSV,
        "--src_root", SRC_ROOT,
    ]

    if USE_FOLDS:
        cmd.append("--use_folds")

    if USE_GPU:
        cmd.append("--use_gpu")

    print("Running evaluation command:")
    print(" ".join(cmd))
    print()

    # Run the evaluation and display detailed logs on failure
    try:
        result = subprocess.run(
            cmd,
            cwd=str(REPO_ROOT),
            check=False,
            capture_output=True,
            text=True,
        )
        if result.stdout:
            print(result.stdout)
        if result.returncode == 0:
            print("\n Evaluation completed successfully!")
        else:
            if result.stderr:
                print("--- STDERR ---")
                print(result.stderr)
            print(f"\n Evaluation exited with code {result.returncode}")
    except Exception as e:
        print(f"Error running evaluation: {e}")

## Step 4: Compute Correlations

After evaluation, compute correlation statistics and generate plots.

In [ ]:
# Build correlation command
cmd = [
    sys.executable, "correlation_VP.py",
    "-m", MODEL_NAME,
    "-v", str(VIEWS),
    "-vm", VIEW_METHOD,
    "-rm", RENDER_METHOD,
    "-db", DATABASE,
    "--out_root", OUT_ROOT,
]

if USE_FOLDS:
    cmd.append("--use_folds")

print("Running correlation command:")
print(" ".join(cmd))
print()

# Run the correlation and display detailed logs on failure
try:
    result = subprocess.run(
        cmd,
        cwd=str(REPO_ROOT),
        check=False,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode == 0:
        print("\n Correlation computation completed!")
    else:
        if result.stderr:
            print("--- STDERR ---")
            print(result.stderr)
        print(f"\n Correlation exited with code {result.returncode}")
except Exception as e:
    print(f"Error running correlation: {e}")

## Step 5: Review Results

Check the output files generated by the evaluation and correlation steps.

In [ ]:
# List output files
out_path = REPO_ROOT / OUT_ROOT / DATABASE / RENDER_METHOD / VIEW_METHOD / MODEL_NAME / f"{VIEWS}VP"

if out_path.exists():
    print(f"Output files found in:")
    print(f"  {out_path}\n")
    
    # Group by fold
    if USE_FOLDS:
        for fold_idx in range(5):
            fold_dir = out_path / f"fold_k{fold_idx}" / "_METRIC_RESULTS_TESTSET_"
            if fold_dir.exists():
                csv_files = list(fold_dir.glob("*.csv"))
                print(f"  fold_k{fold_idx}: {len(csv_files)} CSV files")
                for csv_file in sorted(csv_files)[:2]:  # Show first 2
                    print(f"    └─ {csv_file.name}")
        
        # Show correlation summary files
        summary_files = [f for f in out_path.glob("*.csv")]
        if summary_files:
            print(f"\n  Correlation summaries:")
            for f in sorted(summary_files):
                print(f"    └─ {f.name}")
    else:
        for f in sorted(out_path.rglob("*")):
            if f.is_file() and f.suffix == ".csv":
                rel_path = f.relative_to(out_path)
                size = f.stat().st_size
                print(f"  └─ {rel_path} ({size} bytes)")
else:
    print(f"Output directory not found: {out_path}")
    print(f"\nDebug: checking intermediate paths...")
    parent = REPO_ROOT / OUT_ROOT
    print(f"  {OUT_ROOT} exists: {parent.exists()}")
    if parent.exists():
        print(f"  Contents: {list(parent.iterdir())}")

## Next Steps

### For multi-fold evaluation:

Use the helper script to run all folds:

```bash
scripts\revalidate_table_qualcompare.bat --dry-run
scripts\revalidate_table_qualcompare.bat
```

### To evaluate on a different dataset:

1. Update the configuration at the top of this notebook
2. Change `SRC_ROOT`, `DATABASE`, `MOS_CSV`, and `TESTLIST_CSV`
3. Re-run the validation and evaluation steps

### For custom CSV formats:

See [scripts/README.md](scripts/README.md) for details on expected CSV formats and how to prepare them from your data.